In [ ]:
# === PARAMETERS ===

N_values = [10, 20, 30, 40, 50, 60, 70]

# Tau range for the high-tau verification plot.
tau_min  = 5.0
tau_max  = 10.0
n_tau    = 25

M_MAX    = 4000

SAFETY   = 1.2

# Where to write the figure.
OUTPUT_PDF = 'figures/plot3_high_tau.pdf'


In [ ]:
# === IMPORTS AND PUBLICATION STYLE ===
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import cm
from scipy.optimize import brentq
from scipy.integrate import quad
from scipy.special import zeta
from scipy.interpolate import interp1d
from pathlib import Path
import time

mpl.rcParams.update({
    'figure.figsize': (7, 4.5),
    'font.size': 11,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman', 'CMU Serif', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'axes.labelsize': 13,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'lines.linewidth': 1.4,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
})

Path(OUTPUT_PDF).parent.mkdir(parents=True, exist_ok=True)
print('Setup complete.')


In [ ]:
# === SPATIAL MATRIX ELEMENTS (R-matrix used in the contact formula) ===
def ho_wavefunctions(M, x):
    """phi_n(x) and phi_n'(x) for n = 0..M-1, underflow-safe.

    A recurrence seeded directly on phi_0 = pi^(-1/4) exp(-x^2/2) would
    underflow to zero in float64 for |x| > 38.6, and with it *every* phi_n
    beyond that radius -- including high-lying levels whose classically
    allowed region reaches well past it.  Carrying an explicit per-point log
    scale keeps the orbitals accurate at any radius; norms and <p^2> agree
    with their analytic values to ~1e-12 relative at M = 8600.
    """
    Nx = len(x)
    ell = -x*x/2.0                      # log scale factor
    a = np.zeros(Nx)                    # phi_hat_{n-1}
    b = np.full(Nx, np.pi**-0.25)       # phi_hat_n
    phi_all = np.zeros((M+1, Nx))
    with np.errstate(under='ignore'):
        phi_all[0] = b*np.exp(ell)
        for n in range(0, M):
            if n == 0:
                c = np.sqrt(2.0)*x*b
            else:
                c = np.sqrt(2.0/(n+1))*x*b - np.sqrt(n/(n+1))*a
            m = np.maximum(np.abs(b), np.abs(c))
            resc = (m > 1e50) | ((m > 0) & (m < 1e-50))
            if np.any(resc):
                s = np.where(resc, m, 1.0)
                ell = ell + np.log(s)
                b = b/s; c = c/s
            a, b = b, c
            phi_all[n+1] = b*np.exp(ell)
    phi = phi_all[:M]; dphi = np.zeros((M, Nx))
    for n in range(M):
        tm = np.sqrt(n/2.0)*phi_all[n-1] if n > 0 else 0.0
        dphi[n] = tm - np.sqrt((n+1)/2.0)*phi_all[n+1]
    return phi, dphi

def setup_R(M_max):
    L = np.sqrt(2.0*M_max)+6.0; Nx = max(4000, 12*M_max)   # 12*M resolves the highest orbitals
    x = np.linspace(-L, L, Nx); dx = x[1]-x[0]
    w = np.full(Nx, dx); w[0]=dx/2; w[-1]=dx/2
    phi, dphi = ho_wavefunctions(M_max, x)
    P = (phi**2*w[None,:])@(dphi**2).T
    v = phi*dphi; Q = (v*w[None,:])@v.T
    R = P-Q
    del phi, dphi, P, Q, v
    return R

print(f'Precomputing R matrix at M_MAX={M_MAX} ...')
t0 = time.time()
R_MAT = setup_R(M_MAX)
print(f'  ready in {time.time()-t0:.1f}s, {R_MAT.nbytes/1e6:.0f} MB')


In [ ]:
# === CANONICAL CONTACT VIA CONTOUR INTEGRAL ===
def contact_CE_contour(N, tau, R, N_theta=None):
    if N_theta is None: N_theta = max(256, 2*N+64)
    M_R = R.shape[0]; M_full = int(N+14*tau*N+20); M = max(M_R, M_full)
    beta_hw = 1.0/(tau*N); q = np.exp(-np.arange(M)*beta_hw)
    log_r_est = (N-0.5)*beta_hw; log_r_max = max(200, log_r_est*2+50)
    def eq(log_r): r=np.exp(log_r); rq=r*q; return np.sum(rq/(1.0+rq))-N
    log_r_star = brentq(eq, -log_r_max, log_r_max); r = np.exp(log_r_star)
    theta = 2.0*np.pi*np.arange(N_theta)/N_theta; z = r*np.exp(1j*theta)
    zq = z[:,None]*q[None,:]; nbar = zq/(1.0+zq)
    log_Xi = np.sum(np.log1p(zq), axis=1)
    w = np.exp(log_Xi - log_Xi[0].real)
    M_use = min(M, M_R); nbar_R = nbar[:,:M_use]
    G = np.sum(nbar_R*(nbar_R@R[:M_use,:M_use]), axis=1)
    phases = np.exp(-1j*N*theta)
    return (2.0/np.pi)*(np.mean(w*G*phases)/np.mean(w*phases)).real


In [ ]:
# === SCALING FUNCTIONS A(tau) AND B(tau) ===
def solve_xi(tau):
    if tau < 1e-10: return 1.0
    def eq(xi):
        if xi/tau > 500: return xi - 1.0
        return tau*np.log(1+np.exp(xi/tau)) - 1.0
    return brentq(eq, -200*max(tau, 1), 10)

def compute_AB(tau, nu=200):
    xi = solve_xi(tau)
    ulim = np.sqrt(max(xi, 0) + 30*tau) + 5
    un, uw = np.polynomial.legendre.leggauss(nu)
    u = ulim*un; wu = ulim*uw
    I0=np.zeros(nu); I2=np.zeros(nu); J0=np.zeros(nu)
    J2=np.zeros(nu); K0=np.zeros(nu); K2=np.zeros(nu)
    for i in range(nu):
        ui=u[i]; ql=np.sqrt(max(xi-ui**2,0)+30*tau)+5
        def ft(q):
            a=(q**2+ui**2-xi)/tau
            if a>500: return 0.
            if a<-500: return 1.
            return 1./(np.exp(a)+1.)
        I0[i],_=quad(lambda q: ft(q),         -ql,ql,limit=500)
        I2[i],_=quad(lambda q: q**2*ft(q),    -ql,ql,limit=500)
        J0[i],_=quad(lambda q: ft(q)*(1-ft(q)),         -ql,ql,limit=500)
        J2[i],_=quad(lambda q: q**2*ft(q)*(1-ft(q)),    -ql,ql,limit=500)
        K0[i],_=quad(lambda q: ft(q)*(1-ft(q))*(1-2*ft(q)),      -ql,ql,limit=500)
        K2[i],_=quad(lambda q: q**2*ft(q)*(1-ft(q))*(1-2*ft(q)), -ql,ql,limit=500)
    A = 2*np.sqrt(2)/np.pi**3 * np.sum(I0*I2*wu)
    V2 = np.sum(J0*wu); V3 = np.sum(K0*wu)
    H = 0.5*((K0*I2+2*J0*J2+I0*K2)/V2 - V3/V2**2*(J0*I2+I0*J2))
    B = -2*np.sqrt(2)/np.pi**2 * np.sum(H*wu)
    return A, B

tau_AB_grid = np.unique(np.concatenate([
    np.linspace(0.5, 4.0, 18),
    np.linspace(4.5, max(15.0, tau_max+2.0), 18),
]))
print(f'Computing A(tau), B(tau) on {len(tau_AB_grid)} tau grid points ...')
t0 = time.time()
A_grid = np.zeros(len(tau_AB_grid))
B_grid = np.zeros(len(tau_AB_grid))
for i, tau in enumerate(tau_AB_grid):
    A_grid[i], B_grid[i] = compute_AB(tau)
print(f'  done in {time.time()-t0:.1f}s')

A_interp = interp1d(tau_AB_grid, A_grid, kind='cubic', fill_value='extrapolate')
B_interp = interp1d(tau_AB_grid, B_grid, kind='cubic', fill_value='extrapolate')


In [ ]:
# === COMPUTE C_N(tau) ON THE HIGH-TAU GRID ===
tau_high = np.linspace(tau_min, tau_max, n_tau)
C_high   = np.zeros((len(N_values), len(tau_high)))

t_total = time.time()
for i, N in enumerate(N_values):
    t0 = time.time()
    for j, tau in enumerate(tau_high):
        C_high[i, j] = contact_CE_contour(N, tau, R_MAT)
    print(f'  N={N:3d}: {time.time()-t0:.1f}s')
print(f'Total: {time.time()-t_total:.0f}s')


In [ ]:
# === FILTER AND PLOT ===
# Apply the M_MAX/M_full safety filter point-by-point.

# Safety factor: only plot (N, tau) pairs with M_MAX >= SAFETY * M_full(N, tau).
# 1.2 ~ 0.05% truncation.  1.0 ~ 1%.  0.8 ~ 5%.
SAFETY   = 1.2

N_arr = np.array(N_values, dtype=int)
M_full_grid = (N_arr[:, None] * (1 + 14*tau_high[None, :]) + 20).astype(int)
mask        = M_MAX >= SAFETY * M_full_grid

fig, ax = plt.subplots()
colors = cm.viridis(np.linspace(0.05, 0.92, len(N_values)))
n_plotted = 0
for i, N in enumerate(N_values):
    if not np.any(mask[i]):
        continue
    tg      = tau_high[mask[i]]
    scaling = A_interp(tg)*N**2.5 + B_interp(tg)*N**1.5
    ax.plot(tg, C_high[i, mask[i]]/scaling, '-o',
            color=colors[i], ms=3.5, label=f'$N={N}$')
    n_plotted += 1

ax.axhline(1.0, color='k', ls='--', alpha=0.5, lw=0.8)
ax.set_xlabel(r'$\tau$')
ax.set_ylabel(r'$\mathcal{R}_N(\tau)$')
ax.legend(loc='best')
ax.set_xlim(tau_min, tau_max)
ax.set_ylim(0.8, 1.05) 


plt.tight_layout()
plt.savefig(OUTPUT_PDF)
plt.show()

print(f'Plot saved to {OUTPUT_PDF}.')
print(f'{n_plotted} of {len(N_values)} N values shown'
      f' (filtered to M_MAX/M_full >= {SAFETY}).')
if n_plotted < len(N_values):
    excluded = [N for i, N in enumerate(N_values) if not np.any(mask[i])]
    print(f'Excluded: {excluded}')
    print(f'  -> raise M_MAX to admit them, or lower tau_max.')
